In [10]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from skimage.transform import resize

In [11]:
DATA_ROOT = "data/gtzan/genres_original/"
SR = 22050
SEGMENT_DURATION = 3
IMG_SIZE = (128,128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [12]:
genres = sorted(os.listdir(DATA_ROOT))
genre_to_index = {g:i for i,g in enumerate(genres)}

filepaths = []
labels = []

for genre in genres:
    genre_folder = os.path.join(DATA_ROOT, genre)
    for file in os.listdir(genre_folder):
        if file.endswith(".wav"):
            filepaths.append(os.path.join(genre_folder, file))
            labels.append(genre_to_index[genre])

filepaths = np.array(filepaths)
labels = np.array(labels)

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(
    filepaths, labels, test_size=0.3, stratify=labels, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

In [14]:
def wav_to_segments(path, label):
    # convert tensor → python string
    path = path.numpy().decode("utf-8")

    y, sr = librosa.load(path, sr=SR)
    samples_per_segment = SR * SEGMENT_DURATION
    num_segments = int(len(y) // samples_per_segment)

    segments = []
    labels_out = []

    for i in range(num_segments):
        start = i * samples_per_segment
        end = start + samples_per_segment
        segment = y[start:end]

        mel = librosa.feature.melspectrogram(
            y=segment,
            sr=sr,
            n_mels=128,
            n_fft=1024,
            hop_length=512
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

        mel_resized = resize(mel_db, IMG_SIZE, mode='reflect', anti_aliasing=True)

        segments.append(mel_resized.astype(np.float32))
        labels_out.append(label.numpy())

    return np.array(segments), np.array(labels_out)

In [15]:
def create_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def process(path, label):
        segs, labs = tf.py_function(
            wav_to_segments,
            [path, label],
            [tf.float32, tf.int64]
        )

        segs.set_shape([None, 128, 128])
        labs.set_shape([None])

        segs = tf.expand_dims(segs, -1)

        return tf.data.Dataset.from_tensor_slices((segs, labs))

    ds = ds.flat_map(process)
    ds = ds.shuffle(2000)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds

train_ds = create_dataset(X_train, y_train)
val_ds   = create_dataset(X_val, y_val)
test_ds  = create_dataset(X_test, y_test)

In [16]:
def build_model():
    inputs = tf.keras.layers.Input(shape=(128,128,1))

    x = tf.keras.layers.Conv2D(32,3,padding='same',activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.Conv2D(64,3,padding='same',activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    x = tf.keras.layers.Conv2D(128,3,padding='same',activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)

model = build_model()
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 128, 128, 1)]     0         
                                                                 
 conv2d_3 (Conv2D)           (None, 128, 128, 32)      320       
                                                                 
 batch_normalization_3 (Batc  (None, 128, 128, 32)     128       
 hNormalization)                                                 
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 64, 64, 32)       0         
 2D)                                                             
                                                                 
 dropout_4 (Dropout)         (None, 64, 64, 32)        0         
                                                                 
 conv2d_4 (Conv2D)           (None, 64, 64, 64)        1849

In [ ]:
logdir = "logs/gtzan/run1"
tb_cb = tf.keras.callbacks.TensorBoard(logdir)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[tb_cb, tf.keras.callbacks.EarlyStopping(patience=7)]
)

Epoch 1/10


In [ ]:
y_true, y_pred = [], []

for x,y in test_ds:
    preds = model.predict(x)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(y.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=genres)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.show()